# Session 3: Structured Outputs & Output Parsing

## Objectives
- Get reliable JSON output from LLMs
- Use JSON mode and structured output schemas
- Parse and validate LLM responses programmatically
- Handle malformed outputs gracefully

**Duration:** 40 minutes | **Level:** Medium

**Why this matters:** Real applications need structured data (not free text) from LLMs â€” for databases, APIs, UI rendering, etc.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"Setup complete! Model: {MODEL}")

## 1. The Problem: Unstructured Output

Without any format enforcement, LLMs return free text that's hard to parse programmatically.

In [ ]:
# Without structure â€” the model returns free text
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Extract: name, email, and company from: 'John Smith from Acme Corp (john@acme.com)'"}],
    temperature=0
)

result = response.choices[0].message.content
print("Free text output:")
print(result)
print(f"\nType: {type(result)}")
# This is just a string â€” hard to use in code!

## 2. JSON Mode

Set `response_format={"type": "json_object"}` to force the model to output valid JSON.

**Important:** You must also mention "JSON" in your prompt when using JSON mode.

In [ ]:
# JSON mode â€” guarantees valid JSON output
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Extract contact information. Respond in JSON format."},
        {"role": "user", "content": "John Smith from Acme Corp (john@acme.com)"}
    ],
    response_format={"type": "json_object"},
    temperature=0
)

result = response.choices[0].message.content
print("JSON string:")
print(result)

# Now we can parse it into a Python dictionary!
data = json.loads(result)
print(f"\nParsed data: {data}")
print(f"Name: {data.get('name')}")
print(f"Email: {data.get('email')}")

## 3. Structured Outputs with JSON Schema

JSON mode guarantees valid JSON, but the **structure** can vary.
With `json_schema`, you define the **exact** schema the model must follow.

This gives you:
- Guaranteed field names
- Correct data types
- Required vs optional fields

In [ ]:
# Define a strict schema for movie reviews
movie_review_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "movie_review_analysis",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "movie_title": {"type": "string"},
                "sentiment": {"type": "string", "enum": ["positive", "negative", "mixed"]},
                "rating": {"type": "number"},
                "key_themes": {"type": "array", "items": {"type": "string"}},
                "summary": {"type": "string"}
            },
            "required": ["movie_title", "sentiment", "rating", "key_themes", "summary"],
            "additionalProperties": False
        }
    }
}

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Analyze movie reviews and extract structured data."},
        {"role": "user", "content": """The new Dune movie was visually stunning with incredible cinematography.
        The acting was superb, especially Timothee Chalamet. However, the pacing
        was slow in the middle. Overall a great sci-fi epic. 8/10."""}
    ],
    response_format=movie_review_schema,
    temperature=0
)

data = json.loads(response.choices[0].message.content)
print(json.dumps(data, indent=2))

In [ ]:
# Access the structured data programmatically
print(f"Movie: {data['movie_title']}")
print(f"Sentiment: {data['sentiment']}")
print(f"Rating: {data['rating']}")
print(f"Themes: {', '.join(data['key_themes'])}")
print(f"Summary: {data['summary']}")

## 4. Practical Example: Data Extraction Pipeline

Let's build a reusable function that extracts structured data from any text using a schema.

In [ ]:
def extract_structured_data(text, schema, instruction="Extract the requested information."):
    """Extract structured data from text using a JSON schema."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user", "content": text}
        ],
        response_format=schema,
        temperature=0
    )
    return json.loads(response.choices[0].message.content)

# Schema for extracting event information
event_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "event_info",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "event_name": {"type": "string"},
                "date": {"type": "string"},
                "location": {"type": "string"},
                "organizer": {"type": "string"},
                "is_free": {"type": "boolean"}
            },
            "required": ["event_name", "date", "location", "organizer", "is_free"],
            "additionalProperties": False
        }
    }
}

# Extract from natural language text
text = """Join us for the Annual AI Conference on March 15, 2025 at the
Convention Center in San Francisco. Organized by TechForward Inc.
Tickets start at $299."""

event = extract_structured_data(text, event_schema)
print(json.dumps(event, indent=2))

## 5. Processing Multiple Items

A common pattern: process a batch of items and collect structured results.

In [ ]:
# Batch processing with structured output
emails = [
    "Hi, I'd like to cancel my subscription effective immediately. - Mike",
    "When will the new features be released? I'm excited! - Sarah",
    "Your product crashed and I lost all my data. This is unacceptable! - Tom"
]

email_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "email_classification",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "sender": {"type": "string"},
                "intent": {"type": "string", "enum": ["cancellation", "inquiry", "complaint", "feedback"]},
                "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
                "summary": {"type": "string"}
            },
            "required": ["sender", "intent", "urgency", "summary"],
            "additionalProperties": False
        }
    }
}

results = []
for email in emails:
    result = extract_structured_data(email, email_schema, "Classify this customer email.")
    results.append(result)
    print(f"  {result['sender']}: {result['intent']} (urgency: {result['urgency']})")

print(f"\nProcessed {len(results)} emails")

## 6. Error Handling

Always handle potential errors when working with LLM outputs.

In [ ]:
def safe_extract(text, schema, instruction="Extract information."):
    """Extract structured data with error handling."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": text}
            ],
            response_format=schema,
            temperature=0
        )
        
        # Check if the model refused (content filtering)
        if response.choices[0].finish_reason == "content_filter":
            return {"error": "Content was filtered"}
        
        return json.loads(response.choices[0].message.content)
    
    except json.JSONDecodeError as e:
        return {"error": f"Failed to parse JSON: {e}"}
    except Exception as e:
        return {"error": f"API error: {e}"}

# Test with normal input
result = safe_extract(
    "Meeting with Dr. Lisa Park on Friday at 2pm in Room 301",
    event_schema,
    "Extract event details."
)
print(json.dumps(result, indent=2))

## Exercise: Resume Information Extractor

Build a pipeline that extracts structured information from resume/CV text.

In [ ]:
# Define the resume schema
resume_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "resume_data",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "email": {"type": "string"},
                "years_of_experience": {"type": "number"},
                "skills": {"type": "array", "items": {"type": "string"}},
                "education": {"type": "string"},
                "current_role": {"type": "string"}
            },
            "required": ["name", "email", "years_of_experience", "skills", "education", "current_role"],
            "additionalProperties": False
        }
    }
}

resume_text = """Jane Doe | jane.doe@email.com
Senior Machine Learning Engineer at Google (5 years experience)
Previously: Data Scientist at Meta (2 years)
Education: M.S. Computer Science, Stanford University
Skills: Python, PyTorch, TensorFlow, NLP, Computer Vision, MLOps, SQL"""

result = safe_extract(resume_text, resume_schema, "Extract resume information.")
print(json.dumps(result, indent=2))

# Now you can use this data programmatically
print(f"\nCandidate: {result['name']}")
print(f"Experience: {result['years_of_experience']} years")
print(f"Top skills: {', '.join(result['skills'][:3])}")

## Summary

| Method | Use Case |
|--------|----------|
| **JSON mode** | Simple structured output, flexible schema |
| **JSON Schema** | Strict structure with defined fields and types |
| **Error handling** | Production applications that need reliability |

**Key takeaways:**
- Always use `response_format` when you need structured data
- `json_schema` with `strict: True` gives the most reliable results
- Wrap API calls in error handling for production use

**Next session:** Text embeddings and semantic search!